# Titanic Dataset — Cleaning & EDA

**Dataset:** Titanic passenger manifest (the classic Kaggle "Titanic: Machine
Learning from Disaster" training set), sourced here from a public mirror:
`https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv`

**Goal of this notebook:**
1. Load the raw data and inspect its shape, dtypes, and missing values.
2. Clean missing values and fix column types, with every decision documented.
3. Save a clean CSV (`data/titanic_clean.csv`) for downstream use.

The actual cleaning logic lives in `src/clean_titanic.py` so it can be
reused outside a notebook (e.g. in a pipeline or tests). This notebook calls
that function and shows the before/after state.


In [1]:
import sys
sys.path.append("..")  # so we can import src/ when running from notebooks/

import pandas as pd
from src.clean_titanic import clean_titanic

pd.set_option("display.max_columns", None)


## 1. Load raw data

In [2]:
df_raw = pd.read_csv("../data/titanic_raw.csv")
print(df_raw.shape)
df_raw.head()


(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Inspect dtypes and missing values

In [3]:
df_raw.dtypes


PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

In [4]:
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0")


,missing_count,missing_pct
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


**Observations:**
- `Age` is missing for 177 rows (~19.9%).
- `Cabin` is missing for 687 rows (~77.1%) — very sparse.
- `Embarked` is missing for only 2 rows (~0.2%).
- No other columns have missing values.
- `Survived`, `Pclass`, `Sex`, `Embarked` are categorical but stored as
  generic `int64`/`object` dtypes and need to be cast explicitly.


## 3. Clean the data

All logic below is implemented in `src/clean_titanic.py::clean_titanic`.
Summary of the assumptions made there:

| Column | Issue | Decision | Assumption / rationale |
|---|---|---|---|
| `Age` | 177 missing (~20%) | Impute with median age grouped by (`Title` extracted from `Name`, `Pclass`) | Age correlates with title (Master/Miss/Mr/Mrs) and class more than with any single global average; group-wise median is more realistic than one global number. Rows imputed are flagged in `AgeWasMissing`. |
| `Cabin` | 687 missing (~77%) | Extract deck letter into `Deck`; missing → `"Unknown"`; drop raw `Cabin` | Too sparse to impute a specific cabin. Whether a cabin is *known at all* is itself informative (it correlates with class/fare), so we keep that signal explicitly rather than imputing or dropping the column outright. |
| `Embarked` | 2 missing (~0.2%) | Impute with the mode (`"S"`) | Only 2 rows affected; mode imputation is low-risk and avoids discarding otherwise-complete rows. |
| `Fare` | 0 missing here, but guarded | Median fare by `Pclass` if any appear | Defensive default for other snapshots of this dataset that do have missing fares. |
| `Name` | High-cardinality free text | Drop after extracting `Title` | Raw names aren't usable as a feature; the socially meaningful part (title) is preserved. |
| `Ticket` | High-cardinality, inconsistent format | Drop | 681 unique values across 891 rows with no consistent structure. |
| `Pclass`, `Sex`, `Embarked` | Stored as generic types | Cast to `category` | These are genuinely categorical, not continuous numbers or free text. |
| `Survived` | Stored as plain `int64` | Cast to nullable `Int64` | Defensive: keeps room for missing labels (e.g. on the real Kaggle *test* split) without breaking dtype. |


In [5]:
df_clean = clean_titanic(df_raw)
print(df_clean.shape)
df_clean.head()


(891, 14)


,PassengerId,Survived,Pclass,Title,Sex,Age,AgeWasMissing,SibSp,Parch,FamilySize,IsAlone,Fare,Deck,Embarked
0,1,0,3,Mr,male,22.0,False,1,0,2,False,7.2500,Unknown,S
1,2,1,1,Mrs,female,38.0,False,1,0,2,False,71.2833,C,C
2,3,1,3,Miss,female,26.0,False,0,0,1,True,7.9250,Unknown,S
3,4,1,1,Mrs,female,35.0,False,1,0,2,False,53.1000,C,S
4,5,0,3,Mr,male,35.0,False,0,0,1,True,8.0500,Unknown,S


## 4. Verify no missing values remain

In [6]:
assert df_clean.isna().sum().sum() == 0, "Unexpected missing values remain!"
df_clean.isna().sum()


PassengerId      0
Survived         0
Pclass           0
Title            0
Sex              0
Age              0
AgeWasMissing    0
SibSp            0
Parch            0
FamilySize       0
IsAlone          0
Fare             0
Deck             0
Embarked         0
dtype: int64

In [7]:
df_clean.dtypes


PassengerId         int64
Survived            Int64
Pclass           category
Title                 str
Sex              category
Age               float64
AgeWasMissing        bool
SibSp               int64
Parch               int64
FamilySize          int64
IsAlone              bool
Fare              float64
Deck                  str
Embarked         category
dtype: object

## 5. Quick sanity-check EDA on the cleaned data

In [8]:
df_clean.groupby("Pclass", observed=True)["Survived"].mean().rename("survival_rate")


Pclass
1     0.62963
2    0.472826
3    0.242363
Name: survival_rate, dtype: Float64

In [9]:
df_clean.groupby("Sex", observed=True)["Survived"].mean().rename("survival_rate")


Sex
female    0.742038
male      0.188908
Name: survival_rate, dtype: Float64

In [10]:
df_clean["Deck"].value_counts()


Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

## 6. Save cleaned dataset

In [11]:
df_clean.to_csv("../data/titanic_clean.csv", index=False)
print("Saved to ../data/titanic_clean.csv")


Saved to ../data/titanic_clean.csv
